# HRM OHLC Return Maximization

Load the Parquet data for OHLCV and Returns, merge them, and build the PyTorch Dataset & DataLoader for our HRM model.

In [6]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Load the Parquet data
ohlcv_df = pd.read_parquet('/home/saij/ml/numin/data/consolidated_daily_ohlcv.parquet')
returns_df = pd.read_parquet('/home/saij/ml/numin/data/consolidated_daily_returns.parquet')

# Show previews
print("OHLCV Head:")
display(ohlcv_df.tail())

print("\nReturns Head:")
display(returns_df.head())

OHLCV Head:


open                                            \
ticker                    ADANIENT ADANIPORTS APOLLOHOSP ASIANPAINT AXISBANK   
timestamp                                                                      
2026-03-10 00:00:00+05:30   2030.0     1450.0     7865.0     2251.6   1295.2   
2026-03-11 00:00:00+05:30   2006.6     1425.0     7770.5     2290.0   1310.0   
2026-03-12 00:00:00+05:30   1960.0     1402.5     7590.0     2208.0   1243.2   
2026-03-13 00:00:00+05:30   1999.9     1388.7     7556.0     2204.7   1224.0   
2026-03-16 00:00:00+05:30   1961.0     1356.6     7565.0     2185.4   1199.2   

                                                                              \
ticker                    BAJAJ-AUTO BAJAJFINSV BAJFINANCE    BEL BHARTIARTL   
timestamp                                                                      
2026-03-10 00:00:00+05:30     9449.5     1845.1      950.0  465.0     1880.8   
2026-03-11 00:00:00+05:30     9575.0     1861.0      932.2  463.5     1841.0   
2026-03-12 00:00:00+05:30     9280.0     1786.9      880.1  452.0     1797.9   
2026-03-13 00:00:00+05:30     9075.0     1760.1      861.0  451.9     1781.1   
2026-03-16 00:00:00+05:30     8866.0     1738.0      850.1  431.9     1798.0   

                           ...     volume                                    \
ticker                     ...  SUNPHARMA TATACONSUM   TATASTEEL        TCS   
timestamp                  ...                                                
2026-03-10 00:00:00+05:30  ...  3298680.0  1165119.0  25783308.0  4323710.0   
2026-03-11 00:00:00+05:30  ...  2510495.0  1066872.0  34207226.0  2720664.0   
2026-03-12 00:00:00+05:30  ...  2606057.0   961388.0  24413736.0  2752075.0   
2026-03-13 00:00:00+05:30  ...  2554627.0  2855691.0  42683952.0  2466488.0   
2026-03-16 00:00:00+05:30  ...  2452391.0  1527989.0  44789516.0  3807907.0   

                                                                        \
ticker                         TECHM      TITAN        TMPV      TRENT   
timestamp                                                                
2026-03-10 00:00:00+05:30  1069187.0   788850.0  13770080.0   726096.0   
2026-03-11 00:00:00+05:30  1499971.0   454575.0   5949596.0   684748.0   
2026-03-12 00:00:00+05:30  3067753.0  1071366.0   9707654.0   989163.0   
2026-03-13 00:00:00+05:30  1150005.0   868825.0  17073811.0  1215870.0   
2026-03-16 00:00:00+05:30  2062060.0  1013667.0  10394023.0  1303919.0   

                                                  
ticker                    ULTRACEMCO       WIPRO  
timestamp                                         
2026-03-10 00:00:00+05:30   460885.0  12982340.0  
2026-03-11 00:00:00+05:30   231139.0  29199002.0  
2026-03-12 00:00:00+05:30   535730.0  19042475.0  
2026-03-13 00:00:00+05:30   723904.0  12629652.0  
2026-03-16 00:00:00+05:30   728077.0  12631649.0  

[5 rows x 250 columns]


Returns Head:


ticker,ADANIENT,ADANIPORTS,APOLLOHOSP,ASIANPAINT,AXISBANK,BAJAJ-AUTO,BAJAJFINSV,BAJFINANCE,BEL,BHARTIARTL,...,SUNPHARMA,TATACONSUM,TATASTEEL,TCS,TECHM,TITAN,TMPV,TRENT,ULTRACEMCO,WIPRO
timestamp,,,,,,,,,,,,,,,,,,,,,
2000-01-03 00:00:00+05:30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2000-01-04 00:00:00+05:30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-4.000533,0.0,0.0,0.0
2000-01-05 00:00:00+05:30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.372591,0.0,0.0,0.0
2000-01-06 00:00:00+05:30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,4.021220,0.0,0.0,0.0
2000-01-07 00:00:00+05:30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,7.796154,0.0,0.0,0.0


## Merge and Prepare Data
Merge the OHLCV and Returns dataframes into a single continuous dataframe. Ensure the features are ordered `[Open, High, Low, Close, Returns]` as the model expects.

In [ ]:
# Note: Ensure the datasets align on the index (Dates/Tickers) as needed.
# For simplicity, we assume they align or can be concatenated directly for this example.

# Create the specific 5-column dataframe our model requires.
# Data is in wide format (MultiIndex columns: feature x ticker), so convert to long format first.
if isinstance(ohlcv_df.columns, pd.MultiIndex):
    ohlc_cols = [c for c in ['open', 'high', 'low', 'close'] if c in ohlcv_df.columns.get_level_values(0)]
    ohlcv_long = (
        ohlcv_df[ohlc_cols]
        .stack(level=1)
        .rename_axis(index=['timestamp', 'ticker'])
        .reset_index()
    )
else:
    ohlc_cols = ['open', 'high', 'low', 'close'] if 'open' in ohlcv_df.columns else ['Open', 'High', 'Low', 'Close']
    ohlcv_long = ohlcv_df[ohlc_cols].copy().reset_index()
    if 'ticker' not in ohlcv_long.columns:
        ohlcv_long['ticker'] = 'SINGLE'

if isinstance(returns_df.columns, pd.MultiIndex):
    returns_long = (
        returns_df.stack(level=1)
        .rename_axis(index=['timestamp', 'ticker'])
        .reset_index(name='Returns')
    )
else:
    if 'returns' in returns_df.columns:
        returns_long = returns_df[['returns']].copy().reset_index().rename(columns={'returns': 'Returns'})
    elif len(returns_df.columns) == 1:
        only_col = returns_df.columns[0]
        returns_long = returns_df[[only_col]].copy().reset_index().rename(columns={only_col: 'Returns'})
    else:
        returns_long = (
            returns_df.stack()
            .rename_axis(index=['timestamp', 'ticker'])
            .reset_index(name='Returns')
        )

# Merge OHLC and Returns on timestamp/ticker
df = ohlcv_long.merge(returns_long, on=['timestamp', 'ticker'], how='inner')

# Standardize column naming for dataset
rename_map = {'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close'}
df = df.rename(columns=rename_map)

# Keep only model-required columns and drop missing rows
df = df[['Open', 'High', 'Low', 'Close', 'Returns']].dropna()

print(f"Total rows after joining and cleaning: {len(df)}")
display(df.head())

ValueError: Length mismatch: Expected axis has 201 elements, new values have 5 elements

## Dataset and DataLoader
A standard dataset structure slicing our sequences into rolling windows (e.g., past 20 days as input).

In [3]:
class OHLCDataset(Dataset):
    def __init__(self, dataframe, sequence_length=10):
        self.features = dataframe[['Open', 'High', 'Low', 'Close', 'Returns']].values
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.features) - self.sequence_length

    def __getitem__(self, idx):
        x = self.features[idx : idx + self.sequence_length]
        y = self.features[idx + self.sequence_length]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

def create_dataloaders(df, sequence_length=10, batch_size=32, train_split=0.8):
    split_idx = int(len(df) * train_split)
    train_df = df.iloc[:split_idx]
    val_df = df.iloc[split_idx:]
    
    train_loader = DataLoader(OHLCDataset(train_df, sequence_length), batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(OHLCDataset(val_df, sequence_length), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

train_loader, val_loader = create_dataloaders(df, sequence_length=20, batch_size=64)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

KeyError: "['Open' 'High' 'Low' 'Close'] not in index"

## Model Definition (HRM)
A single LSTM mapping past $T$ days of `5 features` towards predicting day $T+1$ OHLC block and Returns vector.

In [ ]:
class HRMModel(nn.Module):
    def __init__(self, input_size=5, hidden_size=64, num_layers=2, output_size=5, dropout=0.2):
        super(HRMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :] # Final timestep output
        out = self.fc1(out)
        out = self.relu(out)
        return self.fc2(out)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HRMModel(input_size=5, hidden_size=64, num_layers=2, output_size=5).to(device)
print(model)

## Custom Loss & Training Loop
Implementing a custom loss that weights MSE against a directional trading term prioritizing matching actual return directions.

In [ ]:
def mkt_return_loss(predictions, targets, alpha=0.5):
    # Base Mean Squared Error
    mse_loss = nn.MSELoss()(predictions, targets)
    
    # Financial alignment (pred & real signs matching is good -> negative diff is good)
    pred_returns = predictions[:, 4]
    actual_returns = targets[:, 4]
    directional_loss = -torch.mean(torch.sign(pred_returns) * actual_returns)
    
    return mse_loss + alpha * directional_loss

def train_hrm(model, train_loader, val_loader, epochs=10, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            preds = model(batch_x)
            
            loss = mkt_return_loss(preds, batch_y, alpha=0.5)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                preds = model(batch_x)
                val_loss += mkt_return_loss(preds, batch_y, alpha=0.5).item()
        val_loss /= len(val_loader)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
# Uncomment to run training:
# train_hrm(model, train_loader, val_loader, epochs=10)